In [9]:
##### IMPORTS

import os
os.environ['picaso_refdata'] = r'C:\Users\Alex\Desktop\Picaso\picaso\reference' # THIS MUST GO BEFORE YOUR IMPORT STATEMENT
os.environ['PYSYN_CDBS'] = r'C:\Users\Alex\Desktop\Picaso\grp\redcat\trds' # This is for the stellar data discussed below.

# General
import numpy as np
import astropy.units as u
import bd_support as sup

from pathlib import Path
from itertools import product

# Picaso and Virga
from picaso import justdoit as jdi
from virga import justdoit as vj

# To see what clouds are availible
# vj.available()

In [10]:
##### CONFIGURATIONS

# Directories
sonor_path  = r'C:\Users\Alex\Desktop\Picaso\data\sonora' # Sonora db
# sonor_path  = '/groups/tkaralidi/pbraunschweig/training_set/profiles/'
virga_path  = r'C:\Users\Alex\Desktop\Picaso\data\virga'  # Virga
# virga_path  = '/home/sa221179/picaso/virga/'
# opaci_path  = None # Opacity db
opaci_path  = r'D:\opacity_500k_for_R5000_egpoutput.db'
output_path = Path(r"C:\Users\Alex\Desktop\Picaso\NN_project\outputs")
# output_path = 'home/al864695/ouputs'

# Constant values
wav_range   = [0.5, 30.0] # microns
# R           = 300        # resolution
R           = 5000        # resolution

In [6]:
##### METHOD BROWN DWARF SPECTRUM=

def bd_spectrum(Teff, gravity):
    """
    Compute a BD emission spectrum with Virga clouds.
    """

    # Opacity & inputs
    opa    = jdi.opannection(wav_range, opaci_path)
    bd     = jdi.inputs(calculation="browndwarf")
    bd.phase_angle(0)
    # Convert log g [cgs] to grav [m s^-2]
    # gravity = 10**logg * 1e-2   # 1 cm s^-2 = 0.01 m s^-2
    bd.gravity(gravity, gravity_unit=u.Unit('m/s**2'))
    bd.sonora(sonor_path, Teff)

    # Calculate
    out    = bd.spectrum(opa, full_output=True)
    
    wn, fl = out["wavenumber"], out["thermal"]  # cm^-1 and erg cm^-2 s^-1 cm^-1
    
    # Regrid in wavenumber space because F is defined per wavenumber
    wn, fl = jdi.mean_regrid(wn, fl, R=R)

    # Convert wavenumber [cm^-1] to wavelength [micron]
    #w_um = 1e4 / wn

    return fl

In [8]:
##### GENERATE AND SAVE SPECTRUM (MARGE format, single-case files)

Teff_s = [1351.6243204380194]      # K
grav_s = [1060.001475631001]
logg_s = [np.log10(grav_s * 100)]  # log g cgs

for Teff_i, grav_i in product(Teff_s, grav_s):

    # Run spectrum
    F_i = bd_spectrum(Teff_i, grav_i)

    # Filename encodes the parameters; ML will parse from name
    fname  = (f"TN{float(Teff_i)}g{float(grav_i)}.npy")
    fpath  = output_path / fname

    np.save(fpath, F_i)